# Workshop 4 - Sistema RAG con NVIDIA NIM + Qdrant

## Instalar dependencias

In [1]:
%pip install openai langchain langchain-huggingface langchain-qdrant \
             qdrant-client sentence-transformers python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


## Fase A — Ingesta (ejecutar una sola vez)

Crea el índice Qdrant en disco. El cliente queda disponible como `qdrant_client` para reutilizarlo en la Fase B sin abrir una segunda instancia.

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

COLLECTION   = "karpathy_qdrant"
PERSIST_PATH = "./db/karpathy_qdrant"

loader = TextLoader("docs/intro-to-llms-karpathy.txt", encoding="utf-8")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)
print(f"✅ {len(docs)} chunks generados")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# Cliente único — se reutiliza en Fase B, NO se cierra aquí
qdrant_client = QdrantClient(path=PERSIST_PATH)

# Crear colección solo si no existe
existing = [c.name for c in qdrant_client.get_collections().collections]
if COLLECTION not in existing:
    qdrant_client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=768, distance=Distance.COSINE),
    )
    vectorstore_ingest = QdrantVectorStore(
        client=qdrant_client,
        collection_name=COLLECTION,
        embedding=embeddings,
    )
    vectorstore_ingest.add_documents(docs)
    print(f"✅ Base de datos vectorial creada en {PERSIST_PATH}")
else:
    print(f"ℹ️  Colección '{COLLECTION}' ya existe, omitiendo ingesta.")

✅ 81 chunks generados


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ℹ️  Colección 'karpathy_qdrant' ya existe, omitiendo ingesta.


## Fase B — Pipeline RAG

Reutiliza el mismo `qdrant_client` abierto en la Fase A. Además, se evaluan tres LLM distintos usando las respecticas APIs de [Nvidia build](https://build.nvidia.com/): Llama, Geemma y Mistral.

### LLAMA

In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from typing import List

from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.language_models.llms import LLM
from langchain_core.documents import Document
from langchain_core.callbacks.manager import CallbackManagerForLLMRun

load_dotenv()

nvidia_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("LLAMA_API_KEY"),
)
MODEL = "meta/llama-4-maverick-17b-128e-instruct"

class NvidiaLLM(LLM):
    model: str = MODEL

    @property
    def _llm_type(self) -> str:
        return "nvidia-nim"

    def _call(self, prompt: str, stop=None,
              run_manager: CallbackManagerForLLMRun = None, **kwargs) -> str:
        response = nvidia_client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
        )
        return response.choices[0].message.content

llm = NvidiaLLM()
print(f"✅ LLM: NVIDIA NIM / {MODEL}")

# Reutilizar el cliente ya abierto en Fase A (sin crear uno nuevo)
vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
print("✅ Vector store cargado desde cliente existente")

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.3},
)

def format_docs(docs: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[Fragmento {i+1}]\n{d.page_content}" for i, d in enumerate(docs)
    )

prompt = ChatPromptTemplate.from_template("""
Eres un asistente experto. Responde la pregunta basándote ÚNICAMENTE
en el contexto proporcionado. Si la información no está en el contexto,
di explícitamente que no la tienes.

CONTEXTO:
{context}

PREGUNTA: {question}

RESPUESTA:
""")

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Prueba rápida
question = "What is retrieval augmented generation?"
answer = rag_chain.invoke(question)
print(f"\nPregunta: {question}")
print(f"Respuesta: {answer.strip()}")

✅ LLM: NVIDIA NIM / meta/llama-4-maverick-17b-128e-instruct
✅ Vector store cargado desde cliente existente

Pregunta: What is retrieval augmented generation?
Respuesta: Según el contexto proporcionado, específicamente en el [Fragmento 1], "retrieval augmented generation" se refiere a una capacidad de los modelos de lenguaje grandes personalizados, como los GPT, para referirse a fragmentos de texto en archivos subidos por el usuario y utilizarlos como información de referencia al generar respuestas. Esto funciona de manera similar a la navegación en Internet, pero en su lugar, el modelo puede "navegar" a través de los archivos subidos y utilizarlos como referencia.

En otras palabras, cuando se suben archivos, el modelo puede hacer referencia a partes del texto contenido en esos archivos y usar esa información para crear respuestas más informadas y precisas.


### Generación de respuestas a las 50 preguntas

In [9]:
import json

with open("docs/questions.json", encoding="utf-8") as f:
    test_questions = json.load(f)

print(f"📋 Respondiendo {len(test_questions)} preguntas...\n")

results = []
for i, item in enumerate(test_questions, 1):
    question = item["question"]
    print(f"  [{i}/{len(test_questions)}] {question[:70]}...")

    source_docs = retriever.invoke(question)
    answer = rag_chain.invoke(question)

    results.append({
        "question": question,
        "answer": answer,
        "contexts": [doc.page_content for doc in source_docs],
    })

with open("llama_rag_output.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("\n✅ Resultados guardados en llama_rag_output.json")

📋 Respondiendo 50 preguntas...

  [1/50] What are some security challenges associated with large language model...


  [2/50] What is the purpose of the base model in the process of developing an ...
  [3/50] What is an adversarial example in the context of large language models...
  [4/50] What are the challenges associated with large language models in the c...
  [5/50] What are the key components of large language models and how do they f...
  [6/50] What is an adversarial example in the context of large language models...
  [7/50] What is the significance of parameters and weights in the functioning ...
  [8/50] What methods does the language model use for data collection and organ...
  [9/50] What is the significance of Meta AI in the development of large langua...
  [10/50] What is the purpose of a universal transferable suffix in the context ...
  [11/50] What is the concept of jailbreaking the model in the context of large ...
  [12/50] What are some capabilities of ChatGPT in handling complex queries and ...
  [13/50] What is the significance of self-improvement in the development of sys...


In [10]:
with open("llama_rag_output.json", "r", encoding="utf-8") as f:
    output = json.load(f)

print(f"Total de preguntas respondidas: {len(output)}")
print("\n--- Ejemplo (primera entrada) ---")
print(json.dumps(output[0], indent=1, ensure_ascii=False))

Total de preguntas respondidas: 50

--- Ejemplo (primera entrada) ---
{
 "question": "What are some security challenges associated with large language models?",
 "answer": "Según el contexto proporcionado, específicamente en el Fragmento 4 y también mencionado en el Fragmento 2, algunos de los desafíos de seguridad asociados con los grandes modelos de lenguaje incluyen:\n\n1. **Jailbreak attacks** (ataques de jailbreak): mencionados en el Fragmento 4, donde se da un ejemplo de cómo un usuario podría intentar manipular el modelo para obtener información o realizar acciones no deseadas.\n2. **Ataques mediante frases desencadenantes** (trigger phrases): mencionados en el Fragmento 2, donde un atacante podría incluir una frase específica en los datos de entrenamiento o en las indicaciones dadas al modelo para hacer que realice acciones indeseables.\n\nAdemás, se menciona en el Fragmento 4 que, así como hubo desafíos de seguridad en la pila de computación original, habrá nuevos desafíos de 

### Geemma

In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from typing import List

from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.language_models.llms import LLM
from langchain_core.documents import Document
from langchain_core.callbacks.manager import CallbackManagerForLLMRun

load_dotenv()

nvidia_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("GEEMMA_API_KEY"),
)
MODEL = "google/gemma-3n-e2b-it"

class NvidiaLLM(LLM):
    model: str = MODEL

    @property
    def _llm_type(self) -> str:
        return "nvidia-nim"

    def _call(self, prompt: str, stop=None,
              run_manager: CallbackManagerForLLMRun = None, **kwargs) -> str:
        response = nvidia_client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
        )
        return response.choices[0].message.content

llm = NvidiaLLM()
print(f"✅ LLM: NVIDIA NIM / {MODEL}")

# Reutilizar el cliente ya abierto en Fase A (sin crear uno nuevo)
vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
print("✅ Vector store cargado desde cliente existente")

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.3},
)

def format_docs(docs: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[Fragmento {i+1}]\n{d.page_content}" for i, d in enumerate(docs)
    )

prompt = ChatPromptTemplate.from_template("""
Eres un asistente experto. Responde la pregunta basándote ÚNICAMENTE
en el contexto proporcionado. Si la información no está en el contexto,
di explícitamente que no la tienes.

CONTEXTO:
{context}

PREGUNTA: {question}

RESPUESTA:
""")

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Prueba rápida
question = "What is retrieval augmented generation?"
answer = rag_chain.invoke(question)
print(f"\nPregunta: {question}")
print(f"Respuesta: {answer.strip()}")

✅ LLM: NVIDIA NIM / google/gemma-3n-e2b-it
✅ Vector store cargado desde cliente existente

Pregunta: What is retrieval augmented generation?
Respuesta: Retrieval augmented generation es como el navegador, pero en lugar de navegar por Internet, ChatGPT puede navegar por los archivos que has subido y usarlos como información de referencia para crear respuestas.


### Generación de respuestas a las 50 preguntas

In [7]:
import json

with open("docs/questions.json", encoding="utf-8") as f:
    test_questions = json.load(f)

print(f"📋 Respondiendo {len(test_questions)} preguntas...\n")

results = []
for i, item in enumerate(test_questions, 1):
    question = item["question"]
    print(f"  [{i}/{len(test_questions)}] {question[:70]}...")

    source_docs = retriever.invoke(question)
    answer = rag_chain.invoke(question)

    results.append({
        "question": question,
        "answer": answer,
        "contexts": [doc.page_content for doc in source_docs],
    })

with open("geemma_rag_output.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("\n✅ Resultados guardados en deepsek_rag_output.json")

📋 Respondiendo 50 preguntas...

  [1/50] What are some security challenges associated with large language model...
  [2/50] What is the purpose of the base model in the process of developing an ...
  [3/50] What is an adversarial example in the context of large language models...
  [4/50] What are the challenges associated with large language models in the c...
  [5/50] What are the key components of large language models and how do they f...
  [6/50] What is an adversarial example in the context of large language models...
  [7/50] What is the significance of parameters and weights in the functioning ...
  [8/50] What methods does the language model use for data collection and organ...
  [9/50] What is the significance of Meta AI in the development of large langua...
  [10/50] What is the purpose of a universal transferable suffix in the context ...
  [11/50] What is the concept of jailbreaking the model in the context of large ...
  [12/50] What are some capabilities of ChatGPT in ha

In [8]:
with open("gemma_rag_output.json", "r", encoding="utf-8") as f:
    output = json.load(f)

print(f"Total de preguntas respondidas: {len(output)}")
print("\n--- Ejemplo (primera entrada) ---")
print(json.dumps(output[0], indent=1, ensure_ascii=False))

Total de preguntas respondidas: 50

--- Ejemplo (primera entrada) ---
{
 "question": "What are some security challenges associated with large language models?",
 "answer": "Según el contexto proporcionado, algunos de los desafíos de seguridad asociados con los modelos de lenguaje grandes son los ataques de \"jailbreak\". Por ejemplo, si se le pregunta a un modelo de lenguaje grande cómo hacer una bomba, podría proporcionar instrucciones.",
 "contexts": [
  "hi everyone, so recently i gave a 30-m talk on large language models, just kind of like an intro talk. um, unfortunately that talk was not recorded, but a lot of people came to me after the talk and they told me that uh they really liked the talk, so i would just i thought i would just re-record it and basically put it up on youtube. so here we go, the busy person's intro to large language models, director scut. okay, so let's begin. first of all, what is a large language model really? well, a large language model is just two files,

### Mistral

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from typing import List

from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.language_models.llms import LLM
from langchain_core.documents import Document
from langchain_core.callbacks.manager import CallbackManagerForLLMRun

load_dotenv()

nvidia_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("MISTRAL_API_KEY"),
)
MODEL = "mistralai/mistral-large-3-675b-instruct-2512"

class NvidiaLLM(LLM):
    model: str = MODEL

    @property
    def _llm_type(self) -> str:
        return "nvidia-nim"

    def _call(self, prompt: str, stop=None,
              run_manager: CallbackManagerForLLMRun = None, **kwargs) -> str:
        response = nvidia_client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
        )
        return response.choices[0].message.content

llm = NvidiaLLM()
print(f"✅ LLM: NVIDIA NIM / {MODEL}")

# Reutilizar el cliente ya abierto en Fase A (sin crear uno nuevo)
vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
print("✅ Vector store cargado desde cliente existente")

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.3},
)

def format_docs(docs: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[Fragmento {i+1}]\n{d.page_content}" for i, d in enumerate(docs)
    )

prompt = ChatPromptTemplate.from_template("""
Eres un asistente experto. Responde la pregunta basándote ÚNICAMENTE
en el contexto proporcionado. Si la información no está en el contexto,
di explícitamente que no la tienes.

CONTEXTO:
{context}

PREGUNTA: {question}

RESPUESTA:
""")

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Prueba rápida
question = "What is retrieval augmented generation?"
answer = rag_chain.invoke(question)
print(f"\nPregunta: {question}")
print(f"Respuesta: {answer.strip()}")

✅ LLM: NVIDIA NIM / mistralai/mistral-large-3-675b-instruct-2512
✅ Vector store cargado desde cliente existente

Pregunta: What is retrieval augmented generation?
Respuesta: Según el contexto proporcionado, **Retrieval-Augmented Generation (RAG)** es un método en el que un modelo de lenguaje (como ChatGPT) puede **referenciar fragmentos específicos de texto** de archivos subidos por el usuario para generar respuestas más precisas y contextualizadas.

En el **Fragmento 1** se explica que, al subir archivos a un GPT personalizado, el sistema utiliza RAG para:
1. **"Buscar" dentro de esos archivos** (en lugar de navegar por internet).
2. **Extraer "chunks" (trozos) de información relevante** de los documentos subidos.
3. **Usar esos fragmentos como referencia** al generar respuestas, mejorando la precisión y evitando alucinaciones (información inventada).

En resumen, RAG combina la capacidad de recuperación de información con la generación de texto para ofrecer respuestas basadas en dato

### Generación de respuestas a las 50 preguntas

In [3]:
import json

with open("docs/questions.json", encoding="utf-8") as f:
    test_questions = json.load(f)

print(f"📋 Respondiendo {len(test_questions)} preguntas...\n")

results = []
for i, item in enumerate(test_questions, 1):
    question = item["question"]
    print(f"  [{i}/{len(test_questions)}] {question[:70]}...")

    source_docs = retriever.invoke(question)
    answer = rag_chain.invoke(question)

    results.append({
        "question": question,
        "answer": answer,
        "contexts": [doc.page_content for doc in source_docs],
    })

with open("mistral_rag_output.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("\n✅ Resultados guardados en mistral_rag_output.json")

📋 Respondiendo 50 preguntas...

  [1/50] What are some security challenges associated with large language model...


  [2/50] What is the purpose of the base model in the process of developing an ...
  [3/50] What is an adversarial example in the context of large language models...
  [4/50] What are the challenges associated with large language models in the c...
  [5/50] What are the key components of large language models and how do they f...
  [6/50] What is an adversarial example in the context of large language models...
  [7/50] What is the significance of parameters and weights in the functioning ...
  [8/50] What methods does the language model use for data collection and organ...
  [9/50] What is the significance of Meta AI in the development of large langua...
  [10/50] What is the purpose of a universal transferable suffix in the context ...
  [11/50] What is the concept of jailbreaking the model in the context of large ...
  [12/50] What are some capabilities of ChatGPT in handling complex queries and ...
  [13/50] What is the significance of self-improvement in the development of sys...


In [4]:
with open("mistral_rag_output.json", "r", encoding="utf-8") as f:
    output = json.load(f)

print(f"Total de preguntas respondidas: {len(output)}")
print("\n--- Ejemplo (primera entrada) ---")
print(json.dumps(output[0], indent=1, ensure_ascii=False))

Total de preguntas respondidas: 50

--- Ejemplo (primera entrada) ---
{
 "question": "What are some security challenges associated with large language models?",
 "answer": "Según el contexto proporcionado, algunos **desafíos de seguridad asociados con los modelos de lenguaje grandes (LLMs)** incluyen:\n\n1. **Ataques de *jailbreak***: Intentos de eludir las restricciones o políticas de seguridad del modelo para generar respuestas dañinas, ilegales o no deseadas (ejemplo mencionado: solicitar instrucciones para fabricar algo peligroso como \"napalm\").\n\n2. **Envenenamiento del conjunto de entrenamiento (*data poisoning*)**:\n   - Los modelos se entrenan con grandes cantidades de texto de internet, donde actores malintencionados podrían insertar datos manipulados (ejemplo del contexto: documentos con *trigger phrases* como \"James Bond\" diseñadas para activar comportamientos indeseables en el modelo).\n   - Esto podría permitir a un atacante controlar parcialmente el comportamiento de